# Régression linéaire

## 1. Chargement

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

df = pd.read_csv("../data/final_metropole/donnees_modeles.csv", dtype={"code_dept": str})

pas_variables = ["date_semaine", "date_cible", "annee", "annee_cible", "semaine", "jeu",
                 "TousAge_allergie_urgences", "cible", "pic", "moy4", "population"]
variables = [c for c in df.columns if c not in pas_variables]

train = df[df["jeu"] == "train"]
val = df[df["jeu"] == "val"]
test = df[df["jeu"] == "test"]
train_val = df[df["jeu"] != "test"]

X_train, y_train = train[variables], train["cible"]
X_val, y_val = val[variables], val["cible"]
X_test, y_test = test[variables], test["cible"]
X_train_val, y_train_val = train_val[variables], train_val["cible"]

## 2. Modèle

In [ ]:
# Standardisation et OneHotEncoder
num_cols = [c for c in variables if c != "code_dept"]

prep = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["code_dept"]),
])

modele = Pipeline([("prep", prep), ("rl", LinearRegression())])

In [3]:
def scores(y, pred):
    return {"R2": round(r2_score(y, pred), 4),
            "RMSE": round(mean_squared_error(y, pred) ** 0.5, 1),
            "MAE": round(mean_absolute_error(y, pred), 1)}

## 3. Validation 2023

In [4]:
modele.fit(X_train, y_train)
scores(y_val, modele.predict(X_val))

{'R2': 0.4486, 'RMSE': 240.1, 'MAE': 174.6}

## 4. Modèle final : train 2021-2023, test 2024

In [5]:
modele.fit(X_train_val, y_train_val)

resultats = {
    "Train (2021-2023)": scores(y_train_val, modele.predict(X_train_val)),
    "Test (2024)": scores(y_test, modele.predict(X_test)),
}
pd.DataFrame(resultats).T

,R2,RMSE,MAE
Train (2021-2023),0.4562,238.3,173.1
Test (2024),0.3964,227.7,169.8
